# DATA 266 Lab 2 - Part 3: multimodal agents
Run all cells on the RTX 4050. The smoke test runs before model downloads. Set `RUN_MODELS=True` for the full diffusion-backed scenario run.

In [ ]:
from pathlib import Path
import subprocess,sys,platform
candidates=[Path.cwd(),*Path.cwd().parents]
ROOT=next((p for p in candidates if (p/'part3_agents').is_dir()),Path('/content/Lab-2'))
if not (ROOT/'part3_agents').is_dir(): subprocess.run(['git','clone','--depth','1','https://github.com/denishatank12/Lab-2.git',str(ROOT)],check=True)
P3=ROOT/'part3_agents'; SEED=266
print('repo:',ROOT)
print('commit:',subprocess.run(['git','-C',str(ROOT),'rev-parse','HEAD'],capture_output=True,text=True).stdout.strip())
print('python:',platform.python_version(),'seed:',SEED)
try:
 import torch; print('torch:',torch.__version__,'gpu:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
except Exception as e: print('GPU diagnostic:',e)

In [ ]:
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(P3/'requirements.txt')],check=True)
print('dependencies ready')

In [ ]:
subprocess.run([sys.executable,str(P3/'src'/'smoke_test.py')],check=True)
subprocess.run([sys.executable,str(P3/'src'/'run_red_team.py'),'--output',str(P3/'outputs'/'red_team_results.json')],check=True)
print('SMOKE AND RED-TEAM GATES PASSED')

In [ ]:
from PIL import Image,ImageDraw
sample=P3/'sample_input.png'
if not sample.exists():
    im=Image.new('RGB',(512,512),(80,130,180)); d=ImageDraw.Draw(im); d.ellipse((150,100,360,310),fill=(220,170,100)); d.rectangle((190,300,320,450),fill=(60,70,120)); im.save(sample)
print('input image:',sample)

In [ ]:
RUN_MODELS=True
args=[sys.executable,str(P3/'src'/'run_scenarios.py'),'--output',str(P3/'outputs'/'scenario_traces.json'),'--image',str(sample)]
if RUN_MODELS: args.append('--enable-models')
lora=ROOT/'part2_diffusion'/'outputs'/'run1'
if lora.exists(): args += ['--lora-path',str(lora)]
subprocess.run(args,check=True)
print('scenario traces and generated outputs written under part3_agents/outputs')